### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [13]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content="<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. I remember that they're known for their ability to mimic human speech, but why do they do that? Maybe it's related to their natural behavior. I think some parrots in the wild might mimic other birds or environmental sounds to communicate. But how does that translate to talking to humans?\n\nI should consider their social nature. Parrots are social animals, right? In the wild, they live in flocks, so communication is essential for things like finding food, avoiding predators, or maintaining social bonds. If they're social, maybe they learn to talk by imitating the sounds around them, including humans. But is there a specific reason they choose to mimic human speech?\n\nI also remember reading that parrots have a syrinx, which is their vocal organ, allowing them to produce a wide range of sounds. That could explain their ability to mimic human words. But why do t

In [14]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

d:\udemy\python\venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [15]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure the parameters are correctly formatted in JSON. No other tools are provided, so this should be the only function call needed.\n', 'tool_calls': [{'id': '1eyk7y4gy', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 154, 'total_tokens': 258, 'completion_time': 0.184043756, 'completion_tokens_details': {'reasoning_tokens': 80}, 'prompt_time': 0.006689405, 'prompt_tokens_details': None, 'queue_time': 0.220285422, 'total_time': 0.190733161}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reas

### Tool Execution Loops

In [16]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is currently sunny.


In [17]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user specified Boston, I need to call this function with "Boston" as the location. I\'ll make sure to format the tool call correctly within the XML tags as instructed.\n', 'tool_calls': [{'id': 'b7a0d0969', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.136542774, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.008877995, 'prompt_tokens_details': None, 'queue_time': 0.051440634, 'total_time': 0.145420769}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_deman